# Run Premise with UVEK 2026

Follow [the setup guide](../docs/development/uvek-colleague-testing.md) first. This notebook uses your regular Brightway directory and the project below. Run the cells in order with the **Python (premise)** kernel. The source database is retained; the scenario is exported under a new name.


In [ ]:
from datetime import datetime
from getpass import getpass

import bw2data as bd
import pandas as pd
from premise import NewDatabase

bd.projects.set_current("bafu-2026-ecospold2-biosphere-310")


## Prepare the scenario

Use REMIND SSP2-NPi for 2050. Enter your Premise IAM decryption key when prompted; it is not saved in this notebook.


In [ ]:
ndb = NewDatabase(
    scenarios=[{"model": "remind", "pathway": "SSP2-NPi", "year": 2050}],
    source_db="BAFU:2026-reimported",
    source_family="uvek",
    source_version="2026",
    biosphere_name="ecoinvent-3.10-biosphere",
    key=getpass("Premise IAM key: "),
    keep_source_db_uncertainty=True,
)


## Apply the updates

This review covers biomass, electricity and heat. Additional inventories are downloaded and prepared at this step. Proxy and omission warnings describe known screening limitations; retain them when discussing results.


In [ ]:
ndb.update(["biomass", "electricity", "heat"])


## Export to Brightway

Each export gets a timestamp so you can keep earlier runs for comparison. The exported database remains available after closing Jupyter.


In [ ]:
export_name = f"BAFU:2026-remind-SSP2-NPi-2050-review-{datetime.now():%Y%m%d-%H%M%S}"
ndb.write_db_to_brightway(name=export_name)


## Explore the exported database

The base electricity markets should cover the 12 REMIND regions and World at high, medium and low voltage. Period-average markets also exist; the table below lists only the base markets. For another session, set `export_name` to the database name printed during export and start from here after running the imports above.


In [ ]:
database = bd.Database(export_name)
print(export_name)
print(f"{len(database):,} activities")
pd.DataFrame(
    [
        {"name": a["name"], "location": a["location"], "unit": a["unit"]}
        for a in database
        if a["name"]
        in {
            "market group for electricity, high voltage",
            "market group for electricity, medium voltage",
            "market group for electricity, low voltage",
        }
    ]
).sort_values(["location", "name"]).reset_index(drop=True)


### EUR generation mix

These are the direct inputs per kWh of high-voltage electricity. Generation shares and network self-consumption appear separately. Change `EUR` below to inspect another region.


In [ ]:
(eur,) = [
    a
    for a in database
    if a["name"] == "market group for electricity, high voltage"
    and a["reference product"] == "electricity, high voltage"
    and a["location"] == "EUR"
]
pd.DataFrame(
    [
        {
            "supplier": e.input["name"],
            "location": e.input.get("location"),
            "amount": e["amount"],
            "unit": e["unit"],
        }
        for e in eur.technosphere()
    ]
).sort_values("amount", ascending=False).reset_index(drop=True)


### NEU low-voltage supply

The upstream medium-voltage market should be in NEU too. Native country grids can remain as handovers to the new regional markets; a country-labelled record does not necessarily retain the old generation mix.


In [ ]:
(neu,) = [
    a
    for a in database
    if a["name"] == "market group for electricity, low voltage"
    and a["reference product"] == "electricity, low voltage"
    and a["location"] == "NEU"
]
pd.DataFrame(
    [
        {
            "supplier": e.input["name"],
            "location": e.input.get("location"),
            "amount": e["amount"],
            "unit": e["unit"],
        }
        for e in neu.technosphere()
    ]
)


### Browse other activities

Use `database.search("heat")`, `database.search("biomass")` or another term, then inspect an activity's `.technosphere()` and `.biosphere()` exchanges. The example lists a few heat activities.


In [ ]:
pd.DataFrame(
    [
        {
            "name": a["name"],
            "product": a["reference product"],
            "location": a["location"],
        }
        for a in database.search("heat", limit=15)
    ]
)


### Calculate an inventory

This calculates the life-cycle inventory for **1 kWh of EUR high-voltage electricity**. It does not calculate an impact score; that requires an installed LCIA method. A successful solve does not establish impact completeness. See the setup guide and implementation audit for the remaining approximations.


In [ ]:
import bw2calc as bc

lca = bc.LCA({eur: 1})
lca.lci()
print("Inventory matrix shape:", lca.inventory.shape)
